# PROD Rerun: Additional Properties Tab vs. `view-details` (v12)

**Purpose:** repeat the Additional-Properties-Tab gap test on **PRODUCTION**
(`datacatalog.lanl.gov`) once the server outage / alias migration is resolved.
The 2026-07-27 run on the temporary `datacatalog-b` server (see v11) answered
the question there:

> Additional-tab property values ARE returned by `view-details`, inside
> `propertyInfo.customTabPropertyMap` (not `summaryPropertyMap`). Item schema is
> uniform across maps; the property-name key is `propertyName`.

**What PROD must confirm (the open items):**

1. The same three `propertyInfo` maps + 16-key item schema exist on PROD.
2. **Data Card** (untestable on `-b` -- nothing populated) behaves the same.
3. Which of PROD's **43 groups** (incl. the 35 DCAT groups, absent on `-b`)
   are `SPECIFIC_CUSTOM_PROPERTY_TAB`, and whether their values also appear in
   `customTabPropertyMap`.

This notebook incorporates every fix discovered in v5-v11: `placeToShow` /
`SUMMARY_TAB` semantics, `elementType="VIEWS"` + required `limit`/`offset`,
the `propertyId -> name` join, the `propertyName` (not `name`) item key, the
three-map union extraction, non-raising `view-details` with 404 diagnosis, and
full-response value search. It also **widens the sample**: up to
`MAX_VIEWS_PER_GROUP` views per non-Summary group.


## Step 0 -- Auth setup (PROD)

**BEFORE RUNNING, confirm with Maxen:**

- Is `datacatalog.lanl.gov` (PROD) fully restored and OAuth-enabled post-migration?
- Is the `REDIRECT_URI` environment variable (the original, pre-outage value)
  valid again, or does the client registration need updating?

No temporary hardcoded overrides remain in this version -- everything reads from
the same 6 environment variables as always. If auth or the sanity check fails,
fall back to the host diagnostic pattern from v5/v6 before changing anything else.


In [1]:
import html
import json
import os
import re
import time

import requests
import truststore
import webview
from requests_oauthlib import OAuth2Session

truststore.inject_into_ssl()  # handles LANL internal TLS certs

required = ["AUTH_FLOW_CLIENT_ID", "AUTH_FLOW_CLIENT_SECRET",
            "REDIRECT_URI", "AUTH_URL", "TOKEN_URL", "SCOPE"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise EnvironmentError(f"Missing environment variables: {missing}")
print("All 6 env vars present \u2713")


class OAuthManager:
    """Complete OAuth manager that handles initial auth AND refresh"""

    def __init__(self, client_id, client_secret, redirect_uri, auth_url, token_url, scope):
        self.client_id = client_id
        self.client_secret = client_secret
        self.redirect_uri = redirect_uri
        self.auth_url = auth_url
        self.token_url = token_url
        self.scope = scope
        self.oauth = None
        self.token = None
        self.access_token = None
        self.refresh_token = None
        self.token_expiry = None

    def authenticate(self):
        self.oauth = OAuth2Session(self.client_id, redirect_uri=self.redirect_uri, scope=self.scope)
        authorization_url, state = self.oauth.authorization_url(self.auth_url)
        print("Authenticating...")
        print(f"[DEBUG] authorization_url = {authorization_url}")
        authorization_response = self._launch_browser_auth(authorization_url)
        if not authorization_response:
            print("\u2717 Authentication failed")
            return False
        self.token = self.oauth.fetch_token(
            self.token_url, authorization_response=authorization_response,
            client_secret=self.client_secret,
        )
        self.access_token = self.token["access_token"]
        self.refresh_token = self.token.get("refresh_token")
        self.token_expiry = time.time() + self.token.get("expires_in", 3600)
        print("\u2713 Authentication successful")
        return True

    def _launch_browser_auth(self, authorization_url):
        authorization_response = None

        def on_loaded():
            nonlocal authorization_response
            current_url = window.get_current_url()
            print(f"[DEBUG] page loaded: {current_url}")
            if current_url and self.redirect_uri in current_url:
                authorization_response = current_url
                print("\u2713 Captured Authorization")
                window.hide()
                time.sleep(2.5)
                window.destroy()

        window = webview.create_window(
            "OAuth Authorization", authorization_url, width=800, height=600,
            resizable=True, on_top=True,
        )
        window.events.loaded += on_loaded
        webview.start(private_mode=True)  # never reuse a cached LANL SSO session
        return authorization_response

    def _is_token_expired(self):
        if not self.token_expiry:
            return True
        return time.time() >= (self.token_expiry - 60)

    def _refresh_access_token(self):
        if not self.refresh_token:
            print("\u26a0 No refresh token available, re-authenticating...")
            return self.authenticate()
        try:
            new_token = self.oauth.refresh_token(
                self.token_url, refresh_token=self.refresh_token,
                client_id=self.client_id, client_secret=self.client_secret,
            )
            self.token = new_token
            self.access_token = new_token["access_token"]
            self.refresh_token = new_token.get("refresh_token", self.refresh_token)
            self.token_expiry = time.time() + new_token.get("expires_in", 3600)
            print("\u2713 Token refreshed successfully")
            return True
        except Exception as e:
            print(f"\u2717 Refresh failed: {e}")
            print("Re-authenticating...")
            return self.authenticate()

    def _ensure_authenticated(self):
        if not self.access_token or self._is_token_expired():
            if self.refresh_token:
                self._refresh_access_token()
            else:
                self.authenticate()

    def get(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.get(url, headers=headers, **kwargs)

    def post(self, url, **kwargs):
        self._ensure_authenticated()
        headers = kwargs.pop("headers", {})
        headers["Authorization"] = f"Bearer {self.access_token}"
        headers["Content-Type"] = headers.get("Content-Type", "application/json")
        return requests.post(url, headers=headers, **kwargs)


# ---------------- PROD configuration ----------------
BASE_URL = "https://datacatalog.lanl.gov/denodo-data-catalog"  # PRODUCTION
TARGET_DB = "dataportal"
SERVER_ID = 1            # confirm unchanged on PROD post-migration
ELEMENT_TYPE = "VIEWS"   # confirmed via Swagger on 2026-07-27
MAX_VIEWS_PER_GROUP = 3  # widened sample vs. the -b run

auth_manager = OAuthManager(
    client_id=os.getenv("AUTH_FLOW_CLIENT_ID"),
    client_secret=os.getenv("AUTH_FLOW_CLIENT_SECRET"),
    redirect_uri=os.getenv("REDIRECT_URI"),
    auth_url=os.getenv("AUTH_URL"),
    token_url=os.getenv("TOKEN_URL"),
    scope=os.getenv("SCOPE"),
)
auth_manager.authenticate()


All 6 env vars present ✓
Authenticating...
[DEBUG] authorization_url = https://idp.lanl.gov/as/authorization.oauth2?response_type=code&client_id=REDACTED&redirect_uri=https%3A%2F%2Fdatacatalog-d.lanl.gov%2Foauth%2F2.0%2FredirectURL.jsp&scope=den-datacat-adm&state=REDACTED
[DEBUG] page loaded: https://weblogin.lanl.gov/login
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://weblogin.lanl.gov/
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
[DEBUG] page loaded: https://datacatalog-d.lanl.gov/oauth/2.0/redirectURL.jsp?code=REDACTED&state=REDACTED
✓ Captured Authorization
✓ Authentication successful


True

## Step 0b -- Auth sanity check

Both known-good endpoints must return 200 before anything else runs. If this
asserts, do NOT continue -- re-run the host diagnostic from v5/v6 and check with
Maxen (401 here = the server is not accepting Bearer tokens, as on
`den-datacat-d` during the outage).


In [2]:
r = auth_manager.get(
    BASE_URL + "/public/api/view-details",
    params={"viewName": "announcement", "databaseName": TARGET_DB, "serverId": SERVER_ID},
)
print("view-details:", r.status_code)

r2 = auth_manager.get(BASE_URL + "/public/api/views", params={"serverId": SERVER_ID})
print("list views:  ", r2.status_code)

assert r.status_code == 200 and r2.status_code == 200, (
    "Token not accepted by PROD -- stop here. Re-run the v5/v6 host diagnostic "
    "and confirm the OAuth setup with Maxen."
)
print("Auth sanity check passed -- OK to continue.")


view-details: 200
list views:   200
Auth sanity check passed -- OK to continue.


## Step 1 -- List all property groups and their `placeToShow`

PROD is expected to show **43 groups** (vs. 9 on `-b`). If it doesn't, record the
actual count -- the 9-vs-43 discrepancy is itself an open question for Maxen.


In [3]:
resp = auth_manager.get(BASE_URL + "/public/api/property-management/groups")
resp.raise_for_status()
groups = resp.json()
print(f"Found {len(groups)} property group(s).  (expected ~43 on PROD; -b had 9)\n")

DISPLAY_LOCATION_FIELD = "placeToShow"   # confirmed 2026-07-27
SUMMARY_VALUE = "SUMMARY_TAB"            # confirmed 2026-07-27

group_details = []
for g in groups:
    r = auth_manager.get(BASE_URL + f"/public/api/property-management/groups/{g['id']}")
    r.raise_for_status()
    group_details.append(r.json())

print("placeToShow per group:")
for d in group_details:
    print(f"  {d.get('name'):<35} -> {d.get(DISPLAY_LOCATION_FIELD)!r}")

non_summary_groups = [d for d in group_details
                      if d.get(DISPLAY_LOCATION_FIELD) != SUMMARY_VALUE]
print(f"\nNon-Summary groups on PROD: {len(non_summary_groups)}")
for d in non_summary_groups:
    print(f"  - {d.get('name')} (id={d.get('id')}, {d.get(DISPLAY_LOCATION_FIELD)!r})")


Found 43 property group(s).  (expected ~43 on PROD; -b had 9)

placeToShow per group:
  Default Group                       -> 'SUMMARY_TAB'
  Additional Information              -> 'SPECIFIC_CUSTOM_PROPERTY_TAB'
  Details                             -> 'SUMMARY_TAB'
  API Information                     -> 'SUMMARY_TAB'
  Duplicate Table                     -> 'SUMMARY_TAB'
  View Details                        -> 'SUMMARY_TAB'
  ODS Details                         -> 'SUMMARY_TAB'
  DCAT                                -> 'SPECIFIC_CUSTOM_PROPERTY_TAB'
  Data Card                           -> 'SPECIFIC_CUSTOM_PROPERTY_TAB'
  AI Portal                           -> 'SUMMARY_TAB'
  DCAT AccessRestriction              -> 'GENERAL_CUSTOM_PROPERTY_TAB'
  DCAT Activity                       -> 'GENERAL_CUSTOM_PROPERTY_TAB'
  DCAT Address                        -> 'GENERAL_CUSTOM_PROPERTY_TAB'
  DCAT Agent                          -> 'GENERAL_CUSTOM_PROPERTY_TAB'
  DCAT Attribution           

## Step 2 -- Find assigned views + populated values per non-Summary group

Uses the confirmed contract: collection endpoint (`elementType="VIEWS"`,
required `limit`/`offset`) to find assigned views; per-element endpoint
(`propertyId`/`visualValue`) joined against the group's property definitions
for names. Samples up to `MAX_VIEWS_PER_GROUP` views per group, preferring
views with populated values.


In [4]:
def get_group_elements(group_id, element_type=ELEMENT_TYPE, limit=200, offset=0):
    resp = auth_manager.get(
        BASE_URL + f"/public/api/property-management/groups/{group_id}/elements/{element_type}",
        params={"serverId": SERVER_ID, "limit": limit, "offset": offset},
    )
    if not resp.ok:
        print(f"[{resp.status_code}] {resp.url}")
        print("body:", resp.text[:500])
    resp.raise_for_status()
    return resp.json()


def get_group_property_defs(group_id):
    resp = auth_manager.get(
        BASE_URL + f"/public/api/property-management/groups/{group_id}/properties",
        params={"serverId": SERVER_ID},
    )
    resp.raise_for_status()
    return resp.json()


def get_element_properties(group_id, element_id):
    resp = auth_manager.get(
        BASE_URL + f"/public/api/property-management/groups/{group_id}"
                   f"/elements/VIEWS/{element_id}/properties",
        params={"serverId": SERVER_ID},
    )
    resp.raise_for_status()
    return resp.json()


samples = []  # list of {"group_name","group_id","view","db","element_id","expected_properties"}

for d in non_summary_groups:
    group_id = d["id"]
    rows = get_group_elements(group_id)
    print(f"--- group '{d.get('name')}' (id={group_id}): {len(rows)} assigned-element row(s) ---")
    if not rows:
        print("  nothing assigned -- skipping\n")
        continue

    id_to_name = {p.get("id"): p.get("name") for p in get_group_property_defs(group_id)}

    # Distinct assigned views (property fields in these rows are null -- ignore them)
    seen = {}
    for row in rows:
        key = (row.get("name"), row.get("databaseName"), row.get("elementId"))
        seen.setdefault(key, True)

    taken = 0
    for (vname, vdb, eid) in seen:
        if taken >= MAX_VIEWS_PER_GROUP:
            break
        props = get_element_properties(group_id, eid)
        populated = {}
        for p in props:
            pname = id_to_name.get(p.get("propertyId"), f"propertyId_{p.get('propertyId')}")
            if p.get("visualValue") not in (None, ""):
                populated[pname] = p.get("visualValue")
        marker = f"{len(populated)} populated" if populated else "nothing populated"
        print(f"  {vdb}.{vname} (elementId={eid}): {marker}")
        samples.append({
            "group_name": d.get("name"), "group_id": group_id,
            "view": vname, "db": vdb or TARGET_DB, "element_id": eid,
            "expected_properties": populated,
        })
        taken += 1
    print()

testable = [s for s in samples if s["expected_properties"]]
print(f"Total samples: {len(samples)}; testable (populated values): {len(testable)}")


--- group 'Additional Information' (id=81): 1 assigned-element row(s) ---
  dataportal.admin_option_type_fvts (elementId=198020): 1 populated

--- group 'DCAT' (id=161): 2 assigned-element row(s) ---
  dataportal.bv_oc_footprints_alldesc_esr (elementId=255582): nothing populated
  dataportal.bv_oc_footprints_esr (elementId=255583): nothing populated

--- group 'Data Card' (id=162): 1 assigned-element row(s) ---
  dataportal.bv_oc_footprints_alldesc_esr (elementId=255582): nothing populated

--- group 'DCAT AccessRestriction' (id=182): 0 assigned-element row(s) ---
  nothing assigned -- skipping

--- group 'DCAT Activity' (id=183): 0 assigned-element row(s) ---
  nothing assigned -- skipping

--- group 'DCAT Address' (id=184): 0 assigned-element row(s) ---
  nothing assigned -- skipping

--- group 'DCAT Agent' (id=185): 0 assigned-element row(s) ---
  nothing assigned -- skipping

--- group 'DCAT Attribution' (id=186): 0 assigned-element row(s) ---
  nothing assigned -- skipping

--- gr

## Step 3 -- `view-details` deep search per sample

For each populated property, three checks against the FULL response, using the
**fixed** extraction (`propertyName`, union-aware):

1. name present in `summaryPropertyMap`
2. name present in `customTabPropertyMap`
3. **value** present anywhere in the serialized response (raw + HTML-stripped)

`gap_confirmed` = any populated value absent from the entire response.


In [5]:
def strip_html_tags(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()


def get_view_details(view_name, db_name):
    resp = auth_manager.get(
        BASE_URL + "/public/api/view-details",
        params={"viewName": view_name, "databaseName": db_name, "serverId": SERVER_ID},
    )
    if resp.ok:
        return resp.status_code, resp.json()
    return resp.status_code, None


_views_index = None

def lookup_view_in_list(view_name):
    global _views_index
    if _views_index is None:
        r = auth_manager.get(BASE_URL + "/public/api/views", params={"serverId": SERVER_ID})
        r.raise_for_status()
        _views_index = {v.get("name"): v for v in r.json()}
    return _views_index.get(view_name)


def names_in_map(view_details, map_name):
    # Item key is "propertyName" (NOT "name") -- confirmed 2026-07-27.
    m = view_details.get("propertyInfo", {}).get(map_name) or {}
    names = set()
    for group_name, props in m.items():
        for prop in props:
            names.add(prop.get("propertyName"))
    names.discard(None)
    return names


results = []
for sample in samples:
    expected = sample["expected_properties"]
    base = {"group": sample["group_name"], "view": sample["view"], "db": sample["db"]}

    if not expected:
        results.append({**base, "status": "INCONCLUSIVE -- nothing populated",
                        "gap_confirmed": None})
        continue

    status_code, details = get_view_details(sample["view"], sample["db"])
    if details is None:
        vinfo = lookup_view_in_list(sample["view"])
        diag = ("view NOT in views list -- stale assignment?" if vinfo is None else
                f"in views list: db={vinfo.get('db')!r}, deleted={vinfo.get('deleted')!r}")
        results.append({**base, "status": f"view-details {status_code}; {diag}",
                        "gap_confirmed": None})
        continue

    details_text = json.dumps(details)
    summary_names = names_in_map(details, "summaryPropertyMap")
    custom_names = names_in_map(details, "customTabPropertyMap")

    per_property = []
    for pname, pval in expected.items():
        val_stripped = strip_html_tags(pval)
        value_found = bool((pval and pval in details_text)
                           or (val_stripped and val_stripped in details_text))
        per_property.append({
            "property": pname,
            "value_preview": (val_stripped or str(pval))[:80],
            "in_summaryPropertyMap": pname in summary_names,
            "in_customTabPropertyMap": pname in custom_names,
            "value_anywhere_in_response": value_found,
        })

    missing = [p["property"] for p in per_property if not p["value_anywhere_in_response"]]
    results.append({**base, "per_property": per_property,
                    "properties_with_value_missing_from_view_details": missing,
                    "gap_confirmed": len(missing) > 0})

for r_ in results:
    print(json.dumps(r_, indent=2)); print()


{
  "group": "Additional Information",
  "view": "admin_option_type_fvts",
  "db": "dataportal",
  "per_property": [
    {
      "property": "More Help for Web Services",
      "value_preview": "https://collaborate.lanl.gov/x/tYV4Cw",
      "in_summaryPropertyMap": false,
      "in_customTabPropertyMap": true,
      "value_anywhere_in_response": true
    }
  ],
  "properties_with_value_missing_from_view_details": [],
  "gap_confirmed": false
}

{
  "group": "DCAT",
  "view": "bv_oc_footprints_alldesc_esr",
  "db": "dataportal",
  "status": "INCONCLUSIVE -- nothing populated",
  "gap_confirmed": null
}

{
  "group": "DCAT",
  "view": "bv_oc_footprints_esr",
  "db": "dataportal",
  "status": "INCONCLUSIVE -- nothing populated",
  "gap_confirmed": null
}

{
  "group": "Data Card",
  "view": "bv_oc_footprints_alldesc_esr",
  "db": "dataportal",
  "status": "INCONCLUSIVE -- nothing populated",
  "gap_confirmed": null
}



## Step 4 -- Verdict

In [6]:
conclusive = [r_ for r_ in results if r_.get("gap_confirmed") is not None]
inconclusive = [r_ for r_ in results if r_.get("gap_confirmed") is None]
any_gap = any(r_["gap_confirmed"] for r_ in conclusive)

for r_ in inconclusive:
    print(f"INCONCLUSIVE  {r_['group']} / {r_['db']}.{r_['view']}: {r_['status']}")
if inconclusive:
    print()

if not results:
    print("No non-Summary groups with assigned views on PROD -- record 'N/A, "
          "checked empirically on PROD' in the contract.")
elif not conclusive:
    print("All samples INCONCLUSIVE on PROD too -- non-Summary groups exist but "
          "nothing populated. Record as such; the -b finding stands as the best "
          "available evidence.")
elif any_gap:
    print("GAP CONFIRMED ON PROD: at least one populated Additional-tab value is "
          "absent from the entire view-details response. denodo_properties MUST "
          "also source from property-management -- this OVERRIDES the -b finding; "
          "update the contract.")
else:
    print("PROD CONFIRMS the -b finding: all populated Additional-tab values are "
          "present in view-details (expected location: customTabPropertyMap). "
          "denodo_properties can rely on view-details alone, extracting the UNION "
          "of summaryPropertyMap + generalTabPropertyMap + customTabPropertyMap "
          "via the 'propertyName' key. Question CLOSED -- update the contract.")


INCONCLUSIVE  DCAT / dataportal.bv_oc_footprints_alldesc_esr: INCONCLUSIVE -- nothing populated
INCONCLUSIVE  DCAT / dataportal.bv_oc_footprints_esr: INCONCLUSIVE -- nothing populated
INCONCLUSIVE  Data Card / dataportal.bv_oc_footprints_alldesc_esr: INCONCLUSIVE -- nothing populated

PROD CONFIRMS the -b finding: all populated Additional-tab values are present in view-details (expected location: customTabPropertyMap). denodo_properties can rely on view-details alone, extracting the UNION of summaryPropertyMap + generalTabPropertyMap + customTabPropertyMap via the 'propertyName' key. Question CLOSED -- update the contract.


## Recording the outcome

Whatever the verdict, capture in the contract / Divya notes:

1. PROD group count and the list of `SPECIFIC_CUSTOM_PROPERTY_TAB` groups
   (especially whether any **DCAT** groups are non-Summary).
2. Whether **Data Card** finally had populated values, and its result.
3. The verdict line from Step 4 verbatim, with the date.
4. If the 9-vs-43 group discrepancy between `-b` and PROD persists, raise it
   with Maxen -- it affects any coverage claims made from dev/test environments.
